# Experiment 6 — persistent training and live monitor

Training runs in a detached **tmux** session on the RunPod pod. You can disconnect from RunPod, SSH, or Jupyter and return later. **Keep the pod running**; stopping or terminating it stops training.

Use the **Python (diffusion)** kernel. Run setup, then launch/resume, then monitor. On reconnect, run setup and monitor again. If the monitor cell is still busy, interrupt its kernel first; this does not stop training.

The launcher resumes the latest Experiment 6 checkpoint with the existing configuration and W&B run. Do not also run training in `experiment_6.ipynb` while this session is active.

Logs are saved to `.training/experiment-6/training.log`. Checkpoints and samples use the existing persistent storage. The monitor refreshes every 5 seconds; interruption pauses only the display.

In [ ]:
from pathlib import Path
import os, re, shlex, subprocess, time
from datetime import datetime, timezone
from IPython.display import clear_output, display, Markdown

ROOT = Path('/workspace/Diffusion')
SESSION = 'experiment-6'
STATE = ROOT / '.training' / SESSION
STATE.mkdir(parents=True, exist_ok=True)
LOG = STATE / 'training.log'
EXIT = STATE / 'exit-code'
PYTHON = ROOT / '.venv' / 'bin' / 'python'

def is_running():
    return subprocess.run(['tmux', 'has-session', '-t', '=' + SESSION],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0

print('Training session:', 'running' if is_running() else 'not running')
print('Log:', LOG)


In [ ]:
# Run once to launch/resume. Safe to rerun while this tmux session is active.
if is_running():
    print('Experiment 6 is already running; use the monitor below.')
else:
    EXIT.unlink(missing_ok=True)
    with LOG.open('a') as f:
        f.write('\n=== Launch ' + datetime.now(timezone.utc).isoformat() + ' ===\n')
    command = shlex.join(['env', 'DIFFUSION_CHECKPOINT_TMPDIR=/dev/shm/diffusion-ckpts',
                          'flock', '-n', str(STATE / 'training.lock'),
                          str(PYTHON), '-u', '-m', 'experiments.run_experiment_6'])
    shell = (command + ' >> ' + shlex.quote(str(LOG)) + ' 2>&1; '
             + 'code=$?; printf "%s\n" "$code" > ' + shlex.quote(str(EXIT))
             + '; exit "$code"')
    subprocess.run(['tmux', 'new-session', '-d', '-s', SESSION, '-c', str(ROOT),
                    shlex.join(['/bin/bash', '-c', shell])], check=True)
    print('Launched Experiment 6 in tmux. It resumes the saved checkpoint automatically.')
    print('You can disconnect from RunPod / SSH / Jupyter while the pod stays running.')


In [ ]:
# Rerun this cell after reconnecting. Interrupting it only stops the display.
ansi = re.compile(r'\x1b\[[0-9;?]*[A-Za-z]')
try:
    while True:
        active = is_running()
        text = ''
        if LOG.exists():
            with LOG.open('rb') as f:
                f.seek(max(0, LOG.stat().st_size - 131072))
                text = ansi.sub('', f.read().decode('utf-8', errors='replace'))
        text = text.rsplit('=== Launch ', 1)[-1]
        lines = [line.strip() for line in text.replace('\r', '\n').splitlines() if line.strip()]
        unique = []
        for line in lines:
            if not unique or line != unique[-1]:
                unique.append(line)
        clear_output(wait=True)
        label = 'RUNNING' if active else 'STOPPED'
        if not active and EXIT.exists():
            code = EXIT.read_text().strip()
            label = 'FINISHED (exit 0)' if code == '0' else 'FAILED (exit ' + code + ')'
        display(Markdown('## Experiment 6 — ' + label))
        print('Updated:', datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'))
        if LOG.exists():
            print('Last log update:', datetime.fromtimestamp(LOG.stat().st_mtime, timezone.utc).strftime('%H:%M:%S UTC'))
        gpu = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                              '--format=csv,noheader'], capture_output=True, text=True)
        print('GPU utilization / used / total:', gpu.stdout.strip() or gpu.stderr.strip())
        urls = re.findall(r'https://wandb.ai/[^\s]+/runs/[a-zA-Z0-9]+', text)
        if urls:
            display(Markdown('[Open live loss charts in W&B](' + urls[-1] + ')'))
        vals = [line for line in unique if ', val:' in line]
        if vals:
            print('Latest validation:', vals[-1])
        print('\nRecent training output:\n' + '\n'.join(unique[-12:]))
        if not active:
            break
        time.sleep(5)
except KeyboardInterrupt:
    print('Monitor paused. The training process continues in tmux.')
